# 📜 The Math-to-NumPy Protocol (v2.1)
*The Universal Translation Layer*

### Step 1: The Dependency Check (The Stop Sign)
* **Objective:** Identify Recursion.
* **Look at:** RHS Indices.
* **Action:**
    * *Recursive* ($x_t$ depends on $x_{t-1}$): **STOP**. Use a Loop.
    * *Independent*: Proceed to Vectorization.

### Step 2: The Global Map (The LHS Lock)
* **Objective:** Define the Tensor Universe.
* **Logic:** Distinguish between indices that stay and indices that vanish.
    * **Free Indices (LHS):** These dictate the permanent axes of your final tensor.
    * **Summation Indices (RHS):** Temporary axes used for calculation, then reduced (summed).
* **Example:** $R_{ij} = \sum_k (\dots)$
    * $i \to$ Axis 0
    * $j \to$ Axis 1
    * $k \to$ Axis 2 (to be reduced)

### Step 3: The Shape Classifier
* **Objective:** Handle Iteration Limits.
* **Action:**
    * **Rectangular** ($\sum_{i=1}^N$): Use Slicing (`:`) or standard broadcasting.
    * **Irregular** ($\sum_{j<i}$):
        * *If $i$ is Vectorized:* Use **Masking** (`A * mask` or `np.tril`).
        * *If $i$ is a Loop:* Use **Slicing** (`0:i`).

### Step 4: Vector Representation
* **Policy:** Store data as Flat Arrays $(N,)$.
* **Constraint:** Only promote to Column Vectors $(N,1)$ immediately before a strict Matrix Dot Product (`@`).

### Step 5: The Alignment (Explicit Broadcasting)
* **Objective:** Mechanics over Magic.
* **Action:** Insert `None` (or `np.newaxis`) for missing indices to match the **Global Map**.
* **Target:** If Global Map is $(i, j)$ and term has $j$ only $\to$ `Term[None, :]`.

# 🥋 Vectorization Dojo

**Objective:** Master the "Math-to-NumPy Protocol" for converting any mathematical equation, specifically those related to summation of expressions or series, into vectorized** numpy code
## Challenge 1
### The Equation: Weighted Interaction Matrix
We need to compute matrix $E$, where each element is an interaction between vector $A$ and vector $B$, normalized by the sum of interactions.

$$E_{ij} = \frac{\exp(A_i \cdot B_j)}{\sum_{k} \exp(A_i \cdot B_k)}$$

**Variables:**
* $A$: Vector of size $N$
* $B$: Vector of size $M$

## Step 1: Dependency Check
**Question:** Is there recursion? Does $E_{ij}$ depend on $E_{i-1, j}$?

**Analysis:** No. All indices ($i, j, k$) are independent. 
**Decision:** Proceed to Vectorization.

## Step 2: The Global Map
**Question:** What are the axes of our universe?

**Analysis:**
* **Free Indices (LHS):** $i, j$. These form the output shape.
* **Summation Indices (RHS):** $k$. This axis will be reduced.

**Mapping:**
* $i \to$ Axis 0
* $j \to$ Axis 1

In [13]:
import numpy as np

# Setup Dummy Data
N = 5
M = 3
A = np.random.rand(N) # Shape (N,)
B = np.random.rand(M) # Shape (M,)

## Step 5: The Alignment & Solution

**Logic:**
1. **Align:** We need $A_i$ (Axis 0) and $B_j$ (Axis 1). 
   - $A \to$ `A[:, None]`
   - $B \to$ `B[None, :]`
2. **Numerator:** Compute the interaction grid $(N, M)$.
3. **Denominator:** Compute the interaction grid using $B_k$, sum over $k$ (Axis 1), and **reshape** to maintain alignment.

In [14]:
# 1. The Numerator
# Map: (i, j) -> (Axis 0, Axis 1)
interaction = A[:, None] * B[None, :]  # Broadcasts to (N, M)
numerator = np.exp(interaction)

# 2. The Denominator
# We sum over k (which sits in Axis 1 position for B)
# Trap: np.sum reduces dimension to (N,). We must keep it (N, 1) for division.
denominator_sum = np.sum(numerator, axis=1) # Result: (N,)
denominator = denominator_sum[:, None]      # Result: (N, 1)

# 3. Final Calculation
# (N, M) / (N, 1) -> Valid Broadcast
E = numerator / denominator

print(f"Shape of A: {A.shape}")
print(f"Shape of B: {B.shape}")
print(f"Shape of Result E: {E.shape}")

Shape of A: (5,)
Shape of B: (3,)
Shape of Result E: (5, 3)
